# 04 — Explore Well Bore Database

**Phase 1 — Data Discovery and Understanding**

This notebook is exploratory only. It answers:

- What does one row represent?
- How many records exist?
- What fields exist, and what are their data types?
- Are there missing values?
- What date range is covered?
- Are there duplicate records?
- Are there invalid values?

Plus, since the whole point of exploring this file is to evaluate specific
visualization ideas (a well-location map, county breakdown, drilling-year
trend, active/plugged split, depth distribution, wells-per-lease, and
land/water code), this notebook checks whether each one is actually
feasible with the data found here.

Source file: `data/raw/wells/dbf900.ebc` — an EBCDIC-encoded, hierarchical
mainframe extract of RRC's Well Bore System, downloaded from
https://www.rrc.texas.gov/resource-center/research/data-sets-available-for-download/.
Format is documented in
`data/raw/wells/wba091_well-bore-database.pdf` and summarized in
`../docs/data_wells.md`; read that first for the full segment layout.

Key facts recap:
- Fixed **247-byte** binary records, EBCDIC encoded (assumed code page
  037, same assumption as the other three tapes — not yet independently
  validated the way the production tape was, but real decoded lat/long
  values landing squarely inside Texas's bounding box is a strong
  practical signal it's correct here too)
- No delimiters — records must be read in fixed strides
- **28 possible segment types** — a **flat fan-out**, not a deep nested
  tree like production/P-4: every segment type is a direct child of the
  nearest preceding Root (01) record
- This notebook focuses on **three segments**: **Root (01)** (well
  identity, depth, plug status), **Completion (02)** (the
  production/P-4-compatible lease key: district + lease_nbr + well_nbr),
  and **New Location (13)** (WGS84 latitude/longitude)
- The raw file is **7.3 GB / ~29.8M records** — the **biggest file in
  this project**, roughly 2.6× the P-4 tape and 6.7× the production
  tape. This notebook streams through the entire file once (chunked
  reads) to get exact record-type counts and Root/Completion/New
  Location data — no sampling needed for the primary questions.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path("..") / "src"))

from oil_pipeline.extract.wells import RECORD_LENGTH, load_dbf900

RAW_DATA_PATH = Path("..") / "data" / "raw" / "wells" / "dbf900.ebc"
RAW_DATA_PATH.resolve()


WindowsPath('C:/texas-oil-data-platform/data/raw/wells/dbf900.ebc')

## File size & record count

Confirm the file divides evenly into 247-byte records (validates the fixed-length assumption from the format spec) before reading anything.

In [2]:
file_size = RAW_DATA_PATH.stat().st_size
total_records, remainder = divmod(file_size, RECORD_LENGTH)

print(f"File size:        {file_size:,} bytes")
print(f"Record length:    {RECORD_LENGTH} bytes")
print(f"Total records:    {total_records:,}")
print(f"Remainder bytes:  {remainder} (should be 0 for a clean fixed-length file)")


File size:        7,366,305,947 bytes
Record length:    247 bytes
Total records:    29,823,101
Remainder bytes:  0 (should be 0 for a clean fixed-length file)


## What does one row represent?

One physical 247-byte record is **one record of one of 28 possible
types** (see `../docs/data_wells.md`), not one well. Unlike production/P-4,
this tape is a **flat fan-out**: every segment type is a direct child of
the nearest preceding Root (01) record, not a deep multi-level tree. The
file reading and EBCDIC/zoned-decimal decoding logic lives in
`src/oil_pipeline/` (`extract/wells.py`, `utils.py`), not in this
notebook — `load_dbf900` below streams the entire file once, tallying
every record by its 2-byte type key, and parses Root, Completion, and New
Location records as it goes (stamping `api_number` from Root onto its
child Completion/New Location records, since they're linked only by file
position, not a stored key).

## Scan the full file

`load_dbf900` (in `src/oil_pipeline/extract/wells.py`) streams all ~29.8M
records in a single pass. This is the expensive cell in this notebook —
it reads the full 7.3 GB file, the biggest in this project — everything
after it just analyzes the results in memory.

In [3]:
import logging

logging.basicConfig(level=logging.INFO, format="%(message)s")

results = load_dbf900(RAW_DATA_PATH)

key_counts = results["key_counts"]
df_root = results["root"]
df_completion = results["completion"]
df_newloc = results["new_location"]

assert key_counts["count"].sum() == total_records, "scanned count should match the file-size-derived total"


...5,000,000 records scanned


...10,000,000 records scanned


...15,000,000 records scanned


...20,000,000 records scanned


...25,000,000 records scanned


Done: 29,823,101 records scanned


In [4]:
key_counts


,key,count,segment,pct
0,04,3658943,WBRMKS (Well Bore Remarks),12.27
1,09,3259923,WBFORM (Well Bore Formation),10.93
2,06,2749294,WBCASE (Well Bore Casing),9.22
3,24,1739171,WBH15RMK (H-15 Remarks),5.83
4,03,1727733,WBDATE (Well Bore Technical Data Forms File Date),5.79
5,16,1715320,WBPLREC (Well Bore Plugging Record),5.75
6,07,1589133,WBPERF (Well Bore Perforations),5.33
7,10,1414816,WBSQEZE (Well Bore Squeeze),4.74
8,01,1211829,WBROOT (Well Bore Technical Data Root),4.06
9,15,1136348,WBPLRMKS (Well Bore Plugging Remarks),3.81


In [5]:
for name, df in [("Root (01)", df_root), ("Completion (02)", df_completion), ("New Location (13)", df_newloc)]:
    print(f"{name}: {len(df):,} rows")


Root (01): 1,211,829 rows
Completion (02): 1,072,179 rows
New Location (13): 1,018,543 rows


## What fields exist, and what are their data types?

Preview each parsed segment. `latitude`/`longitude` come out as Python
`float` (from `unpack_zoned_decimal`); everything else is `str`.

In [6]:
df_root.head()


,api_number,field_district,res_cnty_code,orig_compl_year,orig_compl_month,orig_compl_day,total_depth,plug_flag,water_land_code
0,00100001,06,001,1963,10,27,00000,N,L
1,00100008,06,001,1950,05,00,00000,Y,L
2,00100037,06,001,0000,00,00,06020,Y,L
3,00100038,00,001,0000,00,00,00000,Y,L
4,00100039,00,001,0000,00,00,00000,Y,L


In [7]:
df_root.info()


<class 'pandas.DataFrame'>
RangeIndex: 1211829 entries, 0 to 1211828
Data columns (total 9 columns):
 #   Column            Non-Null Count    Dtype
---  ------            --------------    -----
 0   api_number        1211829 non-null  str  
 1   field_district    1211829 non-null  str  
 2   res_cnty_code     1211829 non-null  str  
 3   orig_compl_year   1211829 non-null  str  
 4   orig_compl_month  1211829 non-null  str  
 5   orig_compl_day    1211829 non-null  str  
 6   total_depth       1211829 non-null  str  
 7   plug_flag         1211829 non-null  str  
 8   water_land_code   1211829 non-null  str  
dtypes: str(9)
memory usage: 115.6 MB


In [8]:
df_completion.head()


,oil_code,district_code,lease_nbr,well_nbr,active_inactive_code,api_number
0,O,06,04411,1,,00100001
1,G,01,5885,,,00100008
2,O,06,05312,201,,00100037
3,O,06,00154,1,,00100039
4,O,06,00154,2,,00100040


In [9]:
df_newloc.head()


,loc_county,latitude,longitude,api_number
0,001,32.001894,-96.032320,00100001
1,001,31.979880,-96.009797,00100008
2,001,31.557520,-95.548110,00100037
3,038,32.008878,-95.994281,00100038
4,039,32.008864,-95.991405,00100039


In [10]:
df_newloc.info()


<class 'pandas.DataFrame'>
RangeIndex: 1018543 entries, 0 to 1018542
Data columns (total 4 columns):
 #   Column      Non-Null Count    Dtype  
---  ------      --------------    -----  
 0   loc_county  1018543 non-null  str    
 1   latitude    1018543 non-null  float64
 2   longitude   1018543 non-null  float64
 3   api_number  1018543 non-null  str    
dtypes: float64(2), str(2)
memory usage: 41.8 MB


## Are there missing values?

This is a fixed-format binary extract — every field is present in every
record by construction (no free-form nulls). "Missing" here instead means
zero-filled or blank fields. This is also where the map/county
visualization ideas get their first real check: is `field_district`
(Root) actually populated?

In [11]:
print("Null counts across all three tables (should all be 0 — fixed binary layout has no free-form nulls):")
print(df_root.isna().sum())
print(df_completion.isna().sum())
print(df_newloc.isna().sum())
print()
print("Root — field_district == '00' (unpopulated district on Root itself):")
no_district = (df_root["field_district"] == "00").sum()
print(f"{no_district:,} / {len(df_root):,} ({no_district / len(df_root):.1%})")
print()
print("New Location — both latitude and longitude unset (== 0):")
zero_coords = ((df_newloc["latitude"] == 0) & (df_newloc["longitude"] == 0)).sum()
print(f"{zero_coords:,} / {len(df_newloc):,} ({zero_coords / len(df_newloc):.1%})")


Null counts across all three tables (should all be 0 — fixed binary layout has no free-form nulls):
api_number          0
field_district      0
res_cnty_code       0
orig_compl_year     0
orig_compl_month    0
orig_compl_day      0
total_depth         0
plug_flag           0
water_land_code     0
dtype: int64
oil_code                0
district_code           0
lease_nbr               0
well_nbr                0
active_inactive_code    0
api_number              0
dtype: int64
loc_county    0
latitude      0
longitude     0
api_number    0
dtype: int64

Root — field_district == '00' (unpopulated district on Root itself):
674,653 / 1,211,829 (55.7%)

New Location — both latitude and longitude unset (== 0):


6,898 / 1,018,543 (0.7%)


## Feasibility check: county/district breakdown

Root's own `field_district` is unpopulated (`00`) on a majority of wells
(see above) — so Root alone can't reliably support a district
breakdown. But `district_code` on the **Completion** segment (the same
field production/P-4 already use) might be more reliably populated,
since it's tied to an actual oil-lease completion rather than the well
bore in the abstract. Check that here before ruling the district/county
visualization out.

In [12]:
print("Completion — oil_code distribution (this file covers gas wells too):")
print(df_completion["oil_code"].value_counts())
print()

oil_completions = df_completion[df_completion["oil_code"] == "O"]
print(f"Oil completions: {len(oil_completions):,} / {len(df_completion):,}")
print()
print("Completion (oil wells only) — district_code distribution:")
print(oil_completions["district_code"].value_counts().sort_index())
print()
no_district_completion = (oil_completions["district_code"] == "00").sum()
print(f"Oil completions with district_code == '00': {no_district_completion:,} / {len(oil_completions):,}")


Completion — oil_code distribution (this file covers gas wells too):
oil_code
O    777471
G    294708
Name: count, dtype: int64



Oil completions: 777,471 / 1,072,179

Completion (oil wells only) — district_code distribution:
district_code
01     68242
02     29333
03     53767
04     28853
05     12340
06     22186
07     17393
08     66414
09     57429
10    206533
11     86449
13     95619
14     32913
Name: count, dtype: int64

Oil completions with district_code == '00': 0 / 777,471


## What date range is covered? (drilling-year trend feasibility)

`orig_compl_year` on Root is the year a well was originally completed.
Check its distribution and range across the full file.

In [13]:
zero_year = (df_root["orig_compl_year"] == "0000").sum()
print(f"Root records with orig_compl_year == '0000' (unset): {zero_year:,} / {len(df_root):,} ({zero_year / len(df_root):.1%})")
print()

real_years = df_root.loc[df_root["orig_compl_year"] != "0000", "orig_compl_year"].astype(int)
print(f"Records with a real completion year: {len(real_years):,}")
print(f"Year range: {real_years.min()} - {real_years.max()}")
print()
print("Wells completed per decade:")
print((real_years // 10 * 10).value_counts().sort_index())


Root records with orig_compl_year == '0000' (unset): 468,812 / 1,211,829 (38.7%)



Records with a real completion year: 743,017


Year range: 88 - 2026

Wells completed per decade:
orig_compl_year
80           1
90           1
1200         2
1900       177
1910       141
1920      1129
1930      5293
1940      9658
1950     27870
1960     44461
1970     81252
1980    251515
1990     66496
2000     98631
2010    116772
2020     39618
Name: count, dtype: int64


## Feasibility check: map (latitude/longitude)

`new_location` carries `WB-WGS84-LATITUDE`/`LONGITUDE`. Check coverage
and whether the decoded values actually land inside Texas's real
geographic bounds (roughly 25.8°–36.5°N, -106.6°–-93.5°W) — a strong
practical sanity check on both the `cp037` assumption and the
longitude-sign correction applied in `parse_new_location`.

In [14]:
valid_coords = df_newloc[(df_newloc["latitude"] != 0) | (df_newloc["longitude"] != 0)]
print(f"New Location records with a real (non-zero) coordinate: {len(valid_coords):,} / {len(df_newloc):,}")
print()
print(f"Latitude range:  {valid_coords['latitude'].min()} to {valid_coords['latitude'].max()}")
print(f"Longitude range: {valid_coords['longitude'].min()} to {valid_coords['longitude'].max()}")
print()

TEXAS_LAT_RANGE = (25.8, 36.6)
TEXAS_LON_RANGE = (-106.7, -93.4)
outside_texas = valid_coords[
    ~valid_coords["latitude"].between(*TEXAS_LAT_RANGE)
    | ~valid_coords["longitude"].between(*TEXAS_LON_RANGE)
]
print(f"Coordinates outside Texas's approximate bounding box: {len(outside_texas):,} / {len(valid_coords):,}")


New Location records with a real (non-zero) coordinate: 1,011,645 / 1,018,543

Latitude range:  25.8565614 to 36.4990475
Longitude range: -106.5368298 to -93.5315465

Coordinates outside Texas's approximate bounding box: 0 / 1,011,645


## Are there duplicate records?

`api_number` should be unique across Root records — one Root per well
bore. Completion is documented as recurring (a well can have more than
one completion over time, e.g. after a workover), so duplicate
`api_number` values there are expected, not an error — but the
`(district_code, lease_nbr, well_nbr)` combination is what matters for
the wells-per-lease feasibility check.

In [15]:
dup_api = df_root.duplicated(subset=["api_number"]).sum()
print(f"Duplicate api_number values in Root: {dup_api:,} / {len(df_root):,}")
print()

dup_completion_api = df_completion.duplicated(subset=["api_number"]).sum()
print(f"Duplicate api_number values in Completion (expected — recurring segment): {dup_completion_api:,} / {len(df_completion):,}")
print()

print("Feasibility check: wells per lease")
oil_completions = df_completion[df_completion["oil_code"] == "O"]
wells_per_lease = oil_completions.groupby(["district_code", "lease_nbr"])["well_nbr"].nunique()
print(f"Distinct oil leases (district_code, lease_nbr) with at least one completion: {len(wells_per_lease):,}")
print(f"Average distinct well numbers per lease: {wells_per_lease.mean():.2f}")
print(f"Max distinct well numbers on one lease: {wells_per_lease.max()}")
print()
print("Distribution of wells-per-lease:")
print(wells_per_lease.value_counts().sort_index().head(10))


Duplicate api_number values in Root: 0 / 1,211,829



Duplicate api_number values in Completion (expected — recurring segment): 251,989 / 1,072,179

Feasibility check: wells per lease


Distinct oil leases (district_code, lease_nbr) with at least one completion: 224,453
Average distinct well numbers per lease: 3.46
Max distinct well numbers on one lease: 3008

Distribution of wells-per-lease:
well_nbr
1     138624
2      33515
3      14183
4       9551
5       5581
6       3950
7       2906
8       2524
9       1788
10      1444
Name: count, dtype: int64


## Are there invalid values?

Sanity-check values against what the format spec says they should be,
and check the two remaining visualization ideas (depth distribution,
active/plugged split, land/water code) along the way:
- `plug_flag` should be `Y`/`N`
- `water_land_code` should be `I`/`B`/`O`/`L`
- `total_depth` should not have implausible outliers (deepest well ever
  drilled anywhere is ~40,000 ft; Texas's deepest wells run in the
  15,000–20,000+ ft range in the Permian Basin)

In [16]:
invalid_plug_flag = (~df_root["plug_flag"].isin(["Y", "N"])).sum()
invalid_water_land = (~df_root["water_land_code"].isin(["I", "B", "O", "L"])).sum()

print(f"Root — plug_flag not Y/N:              {invalid_plug_flag:,} / {len(df_root):,}")
print(f"Root — water_land_code not I/B/O/L:    {invalid_water_land:,} / {len(df_root):,}")
print()
print("plug_flag distribution (active/plugged feasibility):")
print(df_root["plug_flag"].value_counts())
print()
print("water_land_code distribution:")
print(df_root["water_land_code"].value_counts())
print()

depths = pd.to_numeric(df_root["total_depth"], errors="coerce")
print("total_depth percentiles:")
print(depths.quantile([0.5, 0.9, 0.95, 0.99, 0.999, 0.9999, 1.0]))
print()
implausible_depth = (depths > 40_000).sum()
print(f"Records with total_depth > 40,000 ft (deeper than any well ever drilled): {implausible_depth:,} / {len(depths):,}")


Root — plug_flag not Y/N:              0 / 1,211,829
Root — water_land_code not I/B/O/L:    0 / 1,211,829

plug_flag distribution (active/plugged feasibility):
plug_flag
N    743562
Y    468267
Name: count, dtype: int64

water_land_code distribution:
water_land_code
L    1205883
B       3904
O       1972
I         70
Name: count, dtype: int64



total_depth percentiles:
0.5000     2639.0
0.9000     9960.0
0.9500    11234.0
0.9900    13643.0
0.9990    17950.0
0.9999    22660.0
1.0000    71979.0
Name: total_depth, dtype: float64

Records with total_depth > 40,000 ft (deeper than any well ever drilled): 8 / 1,211,829


## Summary & next steps

Full-file scan of all 29,823,101 records:

- One physical row = one 247-byte record of 1 of 28 types, not one well
  — but unlike production/P-4, this tape is a **flat fan-out**: every
  segment type is a direct child of the nearest preceding Root record,
  no deep multi-level nesting.
- **1,211,829 wells** (Root), **1,072,179 Completion records** (777,471
  oil / 294,708 gas — confirmed zero duplicates by `api_number` on Root;
  Completion's 251,989 duplicate `api_number`s are expected, since it's a
  documented recurring segment), **1,018,543 New Location records**.
- Zero null values (fixed binary layout, as expected). Zero invalid
  `plug_flag` or `water_land_code` values.

### Visualization feasibility verdict

| Idea | Feasible? | Finding |
|---|---|---|
| **Well-location map** (real lat/long points) | **Yes** | 1,011,645 / 1,018,543 (99.3%) New Location records have a real coordinate; **100%** of those land inside Texas's actual bounding box (25.86°–36.50°N, -106.54°–-93.53°W) — strong validation of both `cp037` and the longitude-sign fix |
| **County/district breakdown** | **Yes, but not from Root** | Root's own `field_district` is unpopulated (`00`) on **55.7%** of wells — unusable alone. **Completion's `district_code` is the reliable source**: 0 unpopulated out of 777,471 oil completions, full 13-value spread matching the known district encoding |
| **Wells drilled per year** | **Yes, with cleanup** | 61.3% of Root records have a real completion year, spanning 1920s–2020s with a sensible decade-by-decade shape (peaking in the 1980s at 251,515). Two bad values found: `year=88` and `year=1200` (4 records total) — clear data-entry noise, filter these out rather than plot them |
| **Active vs. plugged wells** | **Yes** | Clean binary split, `plug_flag`: 743,562 `N` (not plugged) / 468,267 `Y` (plugged) — genuinely useful lifecycle metric |
| **Well depth distribution** | **Yes, with an outlier filter** | Median 2,639 ft, 99.9th percentile 17,950 ft — all realistic. Only **8 records** (out of 1.2M) exceed 40,000 ft (deeper than any well ever drilled anywhere) — trivial to filter, doesn't undermine the metric |
| **Wells per lease** | **Yes** | Real signal: average 3.46 distinct wells per lease, but heavily right-skewed (most leases have exactly 1 well: 138,624 of 224,453). One extreme outlier — a lease with **3,008** distinct well numbers — worth capping/flagging rather than letting it dominate a chart |
| **Onshore vs. offshore/bay/inland breakdown** | **Yes** | Clean 4-value split, zero invalid values: Land 1,205,883 (99.5%), Bay 3,904, Offshore 1,972, Inland waterway 70 — heavily land-dominated as expected for Texas, but the other 3 categories are real and non-trivial |

All seven ideas from the original discussion are feasible. The one real
correction to the original plan: **join wells to the rest of this
project via Completion's `district_code`, not Root's** — Root doesn't
reliably carry district information.

### Other things to watch for downstream

- **`WB-OIL-LSE-NBR` is 5 digits, not 6** like the production/P4 tapes'
  lease number — confirmed in `../docs/data_wells.md`. Any join to
  existing `lease_id` values needs zero-padding first.
- **`api_number` is the natural join key between Root, Completion, and
  New Location** — stamped via file position in `load_dbf900`, the same
  positional-hierarchy technique used for the production tape.

The file-reading and decoding logic lives in
`src/oil_pipeline/extract/wells.py` (`load_dbf900`) and
`src/oil_pipeline/utils.py` (`unpack_zoned_decimal`, new in this pass for
the signed zoned-decimal lat/long fields) — matching the pattern from
the other three exploration notebooks. This is Phase 1 (inspect +
document) only; no transform/load code has been written for this dataset
yet.